In [ ]:
import os
import getpass
import kagglehub

os.environ["KAGGLE_USERNAME"] = getpass.getpass("Enter Kaggle Username: ")
os.environ["KAGGLE_KEY"] = getpass.getpass("Enter Kaggle API Key: ")

Enter Kaggle Username: ··········
Enter Kaggle API Key: ··········


# Dataset ASVSpoof 2019

In [ ]:
# Download latest version
path = kagglehub.dataset_download("awsaf49/asvpoof-2019-dataset")

print("Path to dataset files:", path)

100%|██████████| 23.6G/23.6G [18:05<00:00, 23.3MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/awsaf49/asvpoof-2019-dataset/versions/1


In [ ]:
%cd /content

/content


In [ ]:
!ln -s /root/.cache/kagglehub/datasets/awsaf49/asvpoof-2019-dataset/versions/1 /content/asvpoof-2019-dataset

## Criando um dataset balanceado

In [16]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Caminho absoluto direto do cache de download
base_search_path = "/root/.cache/kagglehub/datasets/awsaf49/asvpoof-2019-dataset/versions/1"
train_metadata_path = None
audio_folder = None

# Buscando de forma flexível e insensível a maiúsculas/minúsculas
for root, dirs, files in os.walk(base_search_path, followlinks=True):
    for file in files:
        file_lower = file.lower()
        # Busca mais flexível que encontra o protocolo contendo 'la.cm.train' e terminando com '.txt'
        if "la.cm.train" in file_lower and file_lower.endswith(".txt"):
            train_metadata_path = os.path.join(root, file)

    if root.lower().endswith(os.path.join("asvspoof2019_la_train", "flac").lower()):
        audio_folder = root

if not train_metadata_path:
    raise FileNotFoundError("Não foi possível encontrar o arquivo de protocolo de treino LA no dataset.")
if not audio_folder:
    raise FileNotFoundError("Não foi possível encontrar a pasta de áudios 'ASVspoof2019_LA_train/flac' no dataset.")

print(f"Caminho dos metadados encontrado: {train_metadata_path}")
print(f"Caminho da pasta de áudio encontrado: {audio_folder}")

# Leitura das 5 colunas oficiais do protocolo LA 2019
parsed_rows = []
with open(train_metadata_path, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 5:
            parsed_rows.append([parts[0], parts[1], parts[2], parts[3], parts[4]])

columns = ["speaker_id", "audio_id", "system_id", "attack_id", "label"]
df = pd.DataFrame(parsed_rows, columns=columns)
print(f"Sucesso! {len(df)} linhas carregadas do arquivo de metadados.")

Caminho dos metadados encontrado: /root/.cache/kagglehub/datasets/awsaf49/asvpoof-2019-dataset/versions/1/LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt
Caminho da pasta de áudio encontrado: /root/.cache/kagglehub/datasets/awsaf49/asvpoof-2019-dataset/versions/1/LA/LA/ASVspoof2019_LA_train/flac
Sucesso! 25380 linhas carregadas do arquivo de metadados.


In [19]:
# 1. Verificar a distribuição original de classes (bonafide vs spoof)
print("Distribuição original no dataset:")
print(df['label'].value_counts())

# 2. Definir o tamanho da amostra com base na menor classe
min_class_size = df['label'].value_counts().min()
print(f"\nBalanceando as classes... Cada uma terá exatamente {min_class_size} arquivos.")

# 3. Amostragem balanceada automática (Undersampling)
balanced_df = df.groupby('label').apply(
    lambda x: x.sample(n=min_class_size, random_state=42)
).reset_index(drop=True)

# 4. Divisão Estratificada: Treino (80%) e Teste (20%)
# O 'stratify' garante que a proporção 50/50 de bonafide/spoof se mantenha tanto no treino quanto no teste
train_df, test_df = train_test_split(
    balanced_df,
    test_size=0.20,
    stratify=balanced_df['label'],
    random_state=42
)

print("\n================ DATASET ESTRUTURADO RECONCILIADO ================")
print(f"Total na Amostra Balanceada: {len(balanced_df)} áudios reais.")
print(f"-> Qtd para Treino (X_train): {len(train_df)} | Qtd para Teste (X_test): {len(test_df)}")
print("==================================================================")

# Salvar os arquivos CSV de mapeamento no ambiente local do Colab
train_df.to_csv("my_asvspoof2019_train.csv", index=False)
test_df.to_csv("my_asvspoof2019_test.csv", index=False)


Distribuição original no dataset:
label
spoof       22800
bonafide     2580
Name: count, dtype: int64

Balanceando as classes... Cada uma terá exatamente 2580 arquivos.

================ DATASET ESTRUTURADO RECONCILIADO ================
Total na Amostra Balanceada: 5160 áudios reais.
-> Qtd para Treino (X_train): 4128 | Qtd para Teste (X_test): 1032


/tmp/ipykernel_771/3158897198.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  balanced_df = df.groupby('label').apply(


In [27]:
import librosa
import numpy as np

def extract_audio_features(df_split, src_audio_dir, duration_seconds=5):
    X = []
    y = []

    # Define o número exato de amostras baseado no tempo e na taxa de 16kHz
    target_sr = 16000
    max_samples = target_sr * duration_seconds

    total_files = len(df_split)
    print(f"Iniciando o processamento físico de {total_files} arquivos de áudio CRU...")

    for idx, row in df_split.reset_index(drop=True).iterrows():
        audio_id = row['audio_id']
        file_path = os.path.join(src_audio_dir, f"{audio_id}.flac")

        if os.path.exists(file_path):
            try:
                # 1. Carrega o arquivo de áudio cru a 16kHz
                signal, sr = librosa.load(file_path, sr=target_sr)

                # 2. Padroniza o tamanho do áudio (Corta se for longo, preenche com zeros se for curto)
                if len(signal) > max_samples:
                    signal_ready = signal[:max_samples]  # Corta
                else:
                    signal_ready = np.pad(signal, (0, max_samples - len(signal)), mode='constant')  # Preenche com zeros

                # 3. Salva o sinal cru padronizado
                X.append(signal_ready)

                # Mapeamento binário: 0 para humano (bonafide), 1 para deepfake (spoof)
                y.append(0 if row['label'] == 'bonafide' else 1)

            except Exception as e:
                print(f"Erro ao processar o arquivo {audio_id}: {e}")

        if (idx + 1) % 500 == 0 or (idx + 1) == total_files:
            print(f"Progresso: {idx + 1}/{total_files} arquivos extraídos.")

    return np.array(X), np.array(y)

X_train, y_train = extract_audio_features(train_df, audio_folder, duration_seconds=5)
X_test, y_test = extract_audio_features(test_df, audio_folder, duration_seconds=5)

print("\n================ TENSORES DE ÁUDIO PRONTOS ================")
print(f"Dados de Treino -> X_train (Áudio Cru): {X_train.shape} | y_train (Labels): {y_train.shape}")
print(f"Dados de Teste  -> X_test  (Áudio Cru): {X_test.shape}  | y_test  (Labels): {y_test.shape}")
print("===========================================================")

Iniciando o processamento físico de 4128 arquivos de áudio CRU...
Progresso: 500/4128 arquivos extraídos.
Progresso: 1000/4128 arquivos extraídos.
Progresso: 1500/4128 arquivos extraídos.
Progresso: 2000/4128 arquivos extraídos.
Progresso: 2500/4128 arquivos extraídos.
Progresso: 3000/4128 arquivos extraídos.
Progresso: 3500/4128 arquivos extraídos.
Progresso: 4000/4128 arquivos extraídos.
Progresso: 4128/4128 arquivos extraídos.
Iniciando o processamento físico de 1032 arquivos de áudio CRU...
Progresso: 500/1032 arquivos extraídos.
Progresso: 1000/1032 arquivos extraídos.
Progresso: 1032/1032 arquivos extraídos.

================ TENSORES DE ÁUDIO PRONTOS ================
Dados de Treino -> X_train (Áudio Cru): (4128, 80000) | y_train (Labels): (4128,)
Dados de Teste  -> X_test  (Áudio Cru): (1032, 80000)  | y_test  (Labels): (1032,)


## Salvando os dados já formatados para o modelo

In [29]:
import joblib

pasta_tensores = "./tensores_extraidos"
os.makedirs(pasta_tensores, exist_ok=True)
dataset_completo = {
    "X_train": X_train,
    "y_train": y_train,
    "X_test": X_test,
    "y_test": y_test
}
joblib.dump(dataset_completo, os.path.join(pasta_tensores, "dataset_asvspoof_completo.pkl"), compress=3)

['./tensores_extraidos/dataset_asvspoof_completo.pkl']

In [25]:
from google.colab import drive
import os

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [30]:
#Salvando no google drive

caminho_arquivo_pkl = "/content/drive/MyDrive/Mestrado/Matérias/IA Adersarial/ASVspoof/dataset_asvspoof_completo.pkl"
os.makedirs(caminho_arquivo_pkl, exist_ok=True)
caminho_final_pkl = os.path.join(caminho_arquivo_pkl, "dataset_asvspoof_completo.pkl")
joblib.dump(dataset_completo, caminho_final_pkl, compress=3)
print(f"Dados salvos em {caminho_final_pkl}")

Dados salvos em /content/drive/MyDrive/Mestrado/Matérias/IA Adersarial/ASVspoof/dataset_asvspoof_completo.pkl/dataset_asvspoof_completo.pkl
